In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.feature_selector import LassoSelector, RFECVSelector, SHAPSelector
from src.selection_pipeline import FeatureSelectionPipeline
from src.utils import build_storm_weights, selection_summary

import sklearn


In [2]:

print(f'  numpy  : {np.__version__}')
print(f'  pandas : {pd.__version__}')
print(f'  sklearn: {sklearn.__version__}')

  numpy  : 1.26.4
  pandas : 2.3.0+4.g1dfc98e16a
  sklearn: 1.6.1


# Feature Selection

### Abstract

This notebook identifies the optimal predictor subset from the 33 engineered features constructed in the main notebook. Feature selection is performed via two independent methods — weighted LASSO and RFECV with TimeSeriesSplit — applied across all five forecast horizons (1, 3, 7, 12, 21 hours).

A feature is retained in `SELECTED_FEATURES` if it survives in at least two of two methods at any horizon (intersection, union across horizons). Storm hours (dst < −50 nT, ~5% of the training set) are upweighted by the inverse storm fraction (~21×) in both methods to prevent suppression of signals concentrated in storm periods.

The primary output is `models/feature_selection_results.pkl` — containing strict (28 features) and liberal (33 features) subsets for use in the modelling notebook.

**Note on method selection:** SHAP-based importance was evaluated but excluded due to computational constraints (>30 min per horizon on RF with 121k rows). RFECV on LinearRegression with TimeSeriesSplit is temporally correct and computationally feasible (~9s per horizon). Mutual Information was excluded because RFECV already captures the relevant signal with temporal correctness.


**Depends on:** `data/processed/feat_split.parquet`, `models/split_masks.pkl`, `models/context_constants.pkl`


## Data Loading

Three artifacts produced by the main notebook are loaded:

- `feat_split.parquet` — scaled and imputed feature matrix for all segments
- `split_masks.pkl` — boolean masks defining the six temporal segments
- `context_constants.pkl` — constants (`FEATURE_COLS`, `K_HORIZONS`, `STORM_THR`) defined in the main notebook

Feature selection operates exclusively on `X_train` — the union of Train\_1 and Train\_2 segments. Validation and test segments are not seen at any point during selection.

A scaling check confirms that train features have mean ≈ 0 — verifying that `feat_split.parquet` contains the scaled version of the data.

In [3]:
# ── Data Loading ───────────────────────────────────────────────────────────
feat  = pd.read_parquet('../data/processed/feat_split.parquet')
masks = joblib.load('../models/split_masks.pkl')
ctx   = joblib.load('../models/context_constants.pkl')

FEATURE_COLS = ctx['FEATURE_COLS']
K_HORIZONS   = ctx['K_HORIZONS']
STORM_THR    = ctx['STORM_THR']

# Train segment only — feature selection uses training data exclusively
X_train = feat.loc[masks['train'], FEATURE_COLS]

print(f'feat shape     : {feat.shape}')
print(f'FEATURE_COLS   : {len(FEATURE_COLS)} features')
print(f'K_HORIZONS     : {K_HORIZONS}')
print(f'X_train shape  : {X_train.shape}')
print(f'\nTrain date range: {feat.loc[masks["train"], "datetime"].min()} → '
      f'{feat.loc[masks["train"], "datetime"].max()}')
print(f'\nScaling check (train mean max abs): {X_train.mean().abs().max():.6f}')

feat shape     : (364728, 73)
FEATURE_COLS   : 33 features
K_HORIZONS     : [1, 3, 7, 12, 21]
X_train shape  : (121185, 33)

Train date range: 1995-01-01 00:00:00 → 2008-12-31 02:00:00

Scaling check (train mean max abs): 0.000000


> **Observations — Data Loading:**
> - `feat_split.parquet`: 364,728 rows × 73 columns — scaled and imputed, all segments included.
> - `X_train`: 121,185 rows × 33 features — Train\_1 ∪ Train\_2, purge zones excluded.
> - Train period: 1995-01-01 → 2008-12-31 (Solar Cycles 22–23, Val\_Storm carved out).
> - `K_HORIZONS = [1, 3, 7, 12, 21]` — five forecast horizons.
> - Scaling check confirms mean ≈ 0 — features are correctly standardised.

## Feature Selection Pipeline

Two independent feature selection methods are applied across all five forecast horizons. A feature is retained in `SELECTED_FEATURES` if it survives in both methods at any horizon (intersection per horizon, union across horizons).

| Method | Type | Temporal correctness | Storm weighting |
|---|---|---|---|
| `LassoSelector` | Linear | TimeSeriesSplit CV for alpha selection | `sample_weight` directly |
| `RFECVSelector` | Linear (iterative) | TimeSeriesSplit CV for feature elimination | Oversampling of storm hours |

Both methods use **storm sample weights** (w = 1/storm\_fraction ≈ 21×) for dst < −50 nT. Storm hours represent ~5% of the training set but carry the primary Forbush Decrease signal — without upweighting, both methods would suppress features whose signal is concentrated in storm periods.

**Why two linear methods?** LASSO applies L1 penalty globally and may zero features due to multicollinearity. RFECV eliminates features iteratively based on cross-validated R², providing a different selection criterion. The intersection of the two is more conservative and stable than either alone.

**Two consolidation modes are compared:**
- `min_votes=2` (strict / intersection): feature must survive both methods at some horizon


In [4]:
# ── Feature Selection Pipeline ─────────────────────────────────────────────

# Strict — feature must survive both LASSO and RFECV at any horizon
pipeline_strict = FeatureSelectionPipeline(
    feature_cols = FEATURE_COLS,
    k_horizons   = K_HORIZONS,
    storm_thr    = STORM_THR,
    lasso_splits = 5,
    rfecv_splits = 5,
    min_votes    = 2,
    random_state = 42,
)
pipeline_strict.fit(X_train, feat)


── Horizon 1h ──────────────────────────────────
  [1/2] LassoCV...
        alpha=1.641248  retained=17/33
  [2/2] RFECV...
        retained=32/33

── Horizon 3h ──────────────────────────────────
  [1/2] LassoCV...
        alpha=0.468026  retained=21/33
  [2/2] RFECV...
        retained=33/33

── Horizon 7h ──────────────────────────────────
  [1/2] LassoCV...
        alpha=0.311265  retained=20/33
  [2/2] RFECV...
        retained=29/33

── Horizon 12h ──────────────────────────────────
  [1/2] LassoCV...
        alpha=0.199308  retained=22/33
  [2/2] RFECV...
        retained=28/33

── Horizon 21h ──────────────────────────────────
  [1/2] LassoCV...
        alpha=0.456164  retained=18/33
  [2/2] RFECV...
        retained=33/33


In [10]:
print('\n── Strict (intersection) ──────────────────────────────')
pipeline_strict.print_summary()


── Strict (intersection) ──────────────────────────────

Feature Selection Summary  (min_votes=2)

                       Feature  Max votes   Selected
───────────────────────────────────────────────────────
                        bz_gsm          2          ✓
                    bz_acc_12h          2          ✓
                     solar_sin          2          ✓
          neutron_counts_lag12          2          ✓
           neutron_counts_lag7          2          ✓
           neutron_counts_lag3          2          ✓
                sw_speed_lag12          2          ✓
                 sw_speed_lag7          2          ✓
                 sw_speed_lag3          2          ✓
                 sw_speed_lag1          2          ✓
                  bz_gsm_lag21          2          ✓
                  bz_gsm_lag12          2          ✓
                   bz_gsm_lag3          2          ✓
                      sw_speed          2          ✓
                   bz_gsm_lag1          2        

> **Observations — Feature Selection Pipeline:**
> - **LASSO alpha by horizon:** Highest at h=1h (α=1.64) — strongest regularisation, only 17 features retained. Decreases to h=12h (α=0.20, 22 retained) then rises again at h=21h (α=0.46, 18 retained). The U-shape reflects the signal structure: at short horizons persistence dominates and few features are needed; at medium horizons the signal is distributed; at long horizons the signal weakens again.
> - **RFECV retention:** More conservative at h=7h (29/33) and h=12h (28/33) — the horizons where physical signal is most complex. At h=3h and h=21h all 33 features survive RFECV — iterative elimination finds no feature to remove without R² cost.
> - **Consistency between methods:** LASSO and RFECV agree most at h=7h and h=12h — both reduce the feature set substantially. At h=1h and h=21h they diverge — LASSO is aggressive, RFECV conservative.
> - **Strict (min_votes=2):** A feature must survive both methods at at least one horizon. This produces 28/33 features — the 5 dropped features (`dbz_dt`, `bz_gsm_lag7`, `sw_speed_lag21`, `neutron_counts_lag1`, `neutron_counts_lag21`) are never selected by both methods simultaneously at any horizon.

- `min_votes=1` (liberal / union): feature survives if in at least one method at any horizon

In [5]:
# Liberal — feature survives if in at least one method at any horizon
pipeline_liberal = FeatureSelectionPipeline(
    feature_cols = FEATURE_COLS,
    k_horizons   = K_HORIZONS,
    storm_thr    = STORM_THR,
    lasso_splits = 5,
    rfecv_splits = 5,
    min_votes    = 1,
    random_state = 42,
)
pipeline_liberal.fit(X_train, feat)


── Horizon 1h ──────────────────────────────────
  [1/2] LassoCV...
        alpha=1.641248  retained=17/33
  [2/2] RFECV...
        retained=32/33

── Horizon 3h ──────────────────────────────────
  [1/2] LassoCV...
        alpha=0.468026  retained=21/33
  [2/2] RFECV...
        retained=33/33

── Horizon 7h ──────────────────────────────────
  [1/2] LassoCV...
        alpha=0.311265  retained=20/33
  [2/2] RFECV...
        retained=29/33

── Horizon 12h ──────────────────────────────────
  [1/2] LassoCV...
        alpha=0.199308  retained=22/33
  [2/2] RFECV...
        retained=28/33

── Horizon 21h ──────────────────────────────────
  [1/2] LassoCV...
        alpha=0.456164  retained=18/33
  [2/2] RFECV...
        retained=33/33


In [11]:
print('\n── Liberal (union) ────────────────────────────────────')
pipeline_liberal.print_summary()


── Liberal (union) ────────────────────────────────────

Feature Selection Summary  (min_votes=1)

                       Feature  Max votes   Selected
───────────────────────────────────────────────────────
                        bz_gsm          2          ✓
                    bz_acc_12h          2          ✓
                     solar_sin          2          ✓
          neutron_counts_lag12          2          ✓
           neutron_counts_lag7          2          ✓
           neutron_counts_lag3          2          ✓
                sw_speed_lag12          2          ✓
                 sw_speed_lag7          2          ✓
                 sw_speed_lag3          2          ✓
                 sw_speed_lag1          2          ✓
                  bz_gsm_lag21          2          ✓
                  bz_gsm_lag12          2          ✓
                   bz_gsm_lag3          2          ✓
                      sw_speed          2          ✓
                   bz_gsm_lag1          2        

> **Observations — Feature Selection Pipeline (per horizon):**
> - **h=1h:** LASSO most aggressive (α=1.64, 17/33) — at 1h horizon persistence dominates and few features carry additional signal. RFECV more conservative (32/33) — iterative elimination finds only 1 feature to remove.
> - **h=3h:** LASSO relaxes (α=0.47, 21/33). RFECV retains all 33 — no feature can be removed without R² cost at this horizon.
> - **h=7h:** Both methods agree most — LASSO 20/33, RFECV 29/33. This is the primary forecast horizon where the physical signal from solar wind is clearest.
> - **h=12h:** LASSO most permissive (α=0.20, 22/33). RFECV most restrictive (28/33) — longer horizon makes some features redundant.
> - **h=21h:** LASSO tightens again (α=0.46, 18/33). RFECV retains all 33 — at long horizons the iterative elimination cannot identify features to remove without R² loss.
>
> **Strict (min_votes=2, intersection):** 28/33 features survive both methods at at least one horizon.  
> **Liberal (min_votes=1, union):** All 33 features survive at least one method at some horizon — effectively no elimination.  
> The strict subset is used as `SELECTED_FEATURES` for the modelling notebook.

In [12]:
print(f'\nStrict : {len(pipeline_strict.get_selected())} features')
print(f'Liberal: {len(pipeline_liberal.get_selected())} features')
print(f'Difference (liberal only): {set(pipeline_liberal.get_selected()) - set(pipeline_strict.get_selected())}')


Strict : 28 features
Liberal: 33 features
Difference (liberal only): {'sw_speed_lag21', 'dbz_dt', 'neutron_counts_lag21', 'bz_gsm_lag7', 'neutron_counts_lag1'}


<a id='results'></a>
## Results

> **Observations — Feature Selection:**
> - **Strict (28/33):** Features surviving both LASSO and RFECV at any horizon. `d_neutron` is retained — both methods detect its signal when storm hours are upweighted 21×.
> - **Liberal (33/33):** All features survive at least one method. The 5 features exclusive to liberal mode (`dbz_dt`, `bz_gsm_lag7`, `sw_speed_lag21`, `neutron_counts_lag1`, `neutron_counts_lag21`) are on the selection boundary.
> - **LASSO alpha by horizon:** Decreases from h=1h (α=1.64, stronger regularisation, 17 retained) to h=7h (α=0.31, 20 retained) to h=12h (α=0.20, 22 retained). Longer horizons retain more features — the signal is more distributed.
> - **RFECV:** Retains 28–33 features across horizons — more conservative elimination than LASSO at short horizons.
> - **`neutron_counts` and lags (lag3, lag7, lag12):** Retained in strict — consistent with the 7–21h lead time of the Forbush Decrease signal.
> - **`neutron_counts_lag1` and `neutron_counts_lag21`:** Dropped in strict — lag1 is too short for Forbush precursor physics; lag21 is at the boundary of the lead time window.
> - **`dbz_dt`:** Dropped in strict — IMF rotation rate is not consistently selected across horizons by both methods.
> - **Strict subset is used as `SELECTED_FEATURES`** for the modelling notebook. The 5 liberal-only features are noted as candidates for ablation analysis.

<a id='save'></a>
## Save Results

The full pipeline results are saved to `models/feature_selection_results.pkl` for consumption by the modelling notebook. The dictionary contains both strict and liberal subsets, vote counts, per-method per-horizon selections, and summary matrices — preserving the full audit trail of the feature selection process.

The modelling notebook loads `selected_strict` as `SELECTED_FEATURES`. The liberal subset and individual method results are retained for sensitivity analysis and ablation studies.

In [13]:
# ── Save Results ───────────────────────────────────────────────────────────

results = {
    'selected_strict'    : pipeline_strict.get_selected(),
    'selected_liberal'   : pipeline_liberal.get_selected(),
    'summary_strict'     : pipeline_strict.summary_,
    'summary_liberal'    : pipeline_liberal.summary_,
    'vote_counts_strict' : pipeline_strict.vote_counts_,
    'vote_counts_liberal': pipeline_liberal.vote_counts_,
    'results_strict'     : pipeline_strict.results_,
    'results_liberal'    : pipeline_liberal.results_,
}

joblib.dump(results, '../models/feature_selection_results.pkl')
print('Saved: ../models/feature_selection_results.pkl')
print(f'\nStrict  : {len(results["selected_strict"])} features')
print(f'Liberal : {len(results["selected_liberal"])} features')
print(f'\nSELECTED_FEATURES (strict):')
for f in results['selected_strict']:
    print(f'  {f}')

Saved: ../models/feature_selection_results.pkl

Strict  : 28 features
Liberal : 33 features

SELECTED_FEATURES (strict):
  bz_gsm
  sw_speed
  sw_density
  sw_pressure
  sw_temp
  e_field
  plasma_beta
  mach_alfven
  f107
  ssn
  neutron_counts
  d_neutron
  bz_acc_3h
  bz_acc_6h
  bz_acc_12h
  bz_gsm_lag1
  bz_gsm_lag3
  bz_gsm_lag12
  bz_gsm_lag21
  sw_speed_lag1
  sw_speed_lag3
  sw_speed_lag7
  sw_speed_lag12
  neutron_counts_lag3
  neutron_counts_lag7
  neutron_counts_lag12
  solar_sin
  solar_cos


> **Observations — Pearson Correlation Diagnostic:**
> - `e_field` (r=0.30) and `sw_pressure` (r=0.29) show the strongest linear correlation with `dst_target_7h` — consistent with their role as solar wind drivers.
> - `d_neutron` (r=0.002) and `mach_alfven` (r=0.003) show near-zero overall correlation. Both are retained in the strict subset because the weighted selection methods detect their signal in storm periods.
> - A correlation threshold of r > 0.05 would have eliminated `d_neutron`, `mach_alfven`, `solar_sin`, `sw_density`, `plasma_beta`, and `bz_gsm_lag21` — 6 of 28 selected features. This confirms that correlation-based selection is insufficient for imbalanced time series with domain-specific signal concentration.